# Phase 2: Auto-labeling Data Baru

Memuat model terbaik dari Phase 1, memprediksi label data dari MongoDB `comments_preprocessed`. Output disimpan ke MongoDB.

## 1. Import & Inisialisasi

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Iterator

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer, IDF, NGram, RegexTokenizer,
    StringIndexer, StringIndexerModel, VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from dotenv import load_dotenv
from pymongo import MongoClient

# ============================================================
# CONSTANTS
# ============================================================
SEED = 42
TRAIN_RATIO = 0.80
TEXT_COL = 'text_final'
LABEL_COL = 'sentiment'
VALID_LABELS = ('positif', 'netral', 'negatif')
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(dotenv_path=PROJECT_ROOT / '.env', override=True)
MONGO_URI = os.getenv('MONGO_URI', '').strip()
MONGO_DB  = os.getenv('MONGO_DB', 'analisis_sentimen').strip()
if not MONGO_URI:
    raise ValueError('MONGO_URI belum diisi di .env')
if not MONGO_DB:
    raise ValueError('MONGO_DB belum diisi di .env')

## Collection names
MONGO_LABELED_COLLECTION = 'comments_sentiment'
MONGO_PREPROC_COLLECTION = 'comments_preprocessed'
MONGO_AUTO_LABELED_COLLECTION = 'phase2_auto_labeled'


def create_spark_session(app_name: str = 'sentiment-training') -> SparkSession:
    spark = SparkSession.builder \
        .appName(app_name) \
        .master('local[*]') \
        .config('spark.sql.adaptive.enabled', 'true') \
        .config('spark.driver.memory', '8g') \
        .config('spark.sql.shuffle.partitions', '8') \
        .config('spark.default.parallelism', '8') \
        .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
        .getOrCreate()
    # Fix: Hadoop chmod issue on Windows (winutils.exe not available)
    spark._jsc.hadoopConfiguration().set('fs.file.impl', 'org.apache.hadoop.fs.RawLocalFileSystem')
    return spark
    # Note: sesuaikan driver.memory dengan RAM komputer (8g atau 16g)


def stratified_split(df: DataFrame, ratio: float = TRAIN_RATIO, seed: int = SEED):
    window_spec = Window.partitionBy(LABEL_COL).orderBy(F.rand(seed))
    df_ranked = df.withColumn('_rn', F.row_number().over(window_spec))
    label_counts = {r[LABEL_COL]: r['count'] for r in df.groupBy(LABEL_COL).count().collect()}
    train_cond = None
    test_cond = None
    for lbl in VALID_LABELS:
        cnt = label_counts.get(lbl, 0)
        if cnt == 0:
            continue
        train_limit = int(round(cnt * ratio))
        c_train = (F.col(LABEL_COL) == lbl) & (F.col('_rn') <= train_limit)
        c_test = (F.col(LABEL_COL) == lbl) & (F.col('_rn') > train_limit)
        train_cond = c_train if train_cond is None else train_cond | c_train
        test_cond = c_test if test_cond is None else test_cond | c_test
    train_df = df_ranked.filter(train_cond).drop('_rn').cache()
    test_df = df_ranked.filter(test_cond).drop('_rn').cache()
    return train_df, test_df


def build_pipeline(model_name: str, **kwargs) -> Pipeline:
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol='tokens',
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol='label_index', handleInvalid='keep'
    )
    ngram = NGram(n=2, inputCol='tokens', outputCol='bigrams')
    name = model_name.lower().strip()
    vocab_uni = kwargs.get('vocab_uni', 8000)
    vocab_bi = kwargs.get('vocab_bi', 6000)
    min_df = kwargs.get('min_df', 3.0)
    cv_uni = CountVectorizer(inputCol='tokens', outputCol='uni_feat', vocabSize=vocab_uni, minDF=min_df, minTF=1)
    cv_bi = CountVectorizer(inputCol='bigrams', outputCol='bi_feat', vocabSize=vocab_bi, minDF=min_df, minTF=1)

    if name in ('logistic_regression', 'lr', 'logistic regression'):
        assembler = VectorAssembler(inputCols=['uni_feat', 'bi_feat'], outputCol='raw_feat')
        idf = IDF(inputCol='raw_feat', outputCol='features', minDocFreq=2)
        clf = LogisticRegression(
            featuresCol='features', labelCol='label_index', predictionCol='pred_index',
            maxIter=kwargs.get('max_iter', 300), regParam=kwargs.get('reg_param', 0.05),
            elasticNetParam=kwargs.get('elastic_net', 0.15), family='multinomial', tol=1e-4,
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]
    elif name in ('naive_bayes', 'nb', 'naive bayes'):
        use_bigrams = kwargs.get('use_bigrams', False)
        clf = NaiveBayes(
            featuresCol='features', labelCol='label_index', predictionCol='pred_index',
            modelType='multinomial', smoothing=kwargs.get('smoothing', 0.5),
        )
        if use_bigrams:
            assembler = VectorAssembler(inputCols=['uni_feat', 'bi_feat'], outputCol='features')
            stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, label_indexer, clf]
        else:
            assembler = VectorAssembler(inputCols=['uni_feat'], outputCol='features')
            stages = [tokenizer, cv_uni, assembler, label_indexer, clf]
    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')
    return Pipeline(stages=stages)


def compute_metrics(predictions: DataFrame) -> dict:
    total = predictions.count()
    def _eval(metric):
        return MulticlassClassificationEvaluator(
            labelCol='label_index', predictionCol='pred_index', metricName=metric
        ).evaluate(predictions)
    accuracy = _eval('accuracy')
    f1_weighted = _eval('f1')
    precision_weighted = _eval('weightedPrecision')
    recall_weighted = _eval('weightedRecall')

    cm_rows = {
        (r[LABEL_COL], r['pred_label']): r['cnt']
        for r in predictions.groupBy(LABEL_COL, 'pred_label').agg(F.count('*').alias('cnt')).collect()
    }
    per_class = {}
    macro_p = macro_r = macro_f1 = 0.0
    for lbl in VALID_LABELS:
        tp = cm_rows.get((lbl, lbl), 0)
        fp = sum(cm_rows.get((actual, lbl), 0) for actual in VALID_LABELS if actual != lbl)
        fn = sum(cm_rows.get((lbl, pred), 0) for pred in VALID_LABELS if pred != lbl)
        support = tp + fn
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / support if support > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        per_class[lbl] = {'precision': round(p, 6), 'recall': round(r, 6), 'f1': round(f1, 6), 'support': support}
        macro_p += p; macro_r += r; macro_f1 += f1
    n = len(VALID_LABELS)
    cm_dict = {actual: {pred: cm_rows.get((actual, pred), 0) for pred in VALID_LABELS} for actual in VALID_LABELS}
    return {
        'total_samples': total, 'accuracy': round(accuracy, 6),
        'f1_weighted': round(f1_weighted, 6), 'f1_macro': round(macro_f1 / n, 6),
        'precision_weighted': round(precision_weighted, 6), 'precision_macro': round(macro_p / n, 6),
        'recall_weighted': round(recall_weighted, 6), 'recall_macro': round(macro_r / n, 6),
        'per_class': per_class, 'confusion_matrix': cm_dict,
    }


def map_prediction_labels(df, fitted_pipeline):
    # StringIndexerModel always at stages[-2] for both LR and NB pipelines
    si_model = fitted_pipeline.stages[-2]
    lbls = list(si_model.labels)
    when_expr = None
    for i, lbl in enumerate(lbls):
        cond = F.col('pred_index').cast('int') == i
        when_expr = F.when(cond, F.lit(lbl)) if when_expr is None else when_expr.when(cond, F.lit(lbl))
    return df.withColumn('pred_label', when_expr)


def train_and_evaluate(model_key, model_display, train_df, test_df) -> dict:
    print(f'\n>>> Training: {model_display} ...', flush=True)
    for col_name in ("tokens",):
        if col_name in train_df.columns: train_df = train_df.drop(col_name)
        if col_name in test_df.columns:  test_df = test_df.drop(col_name)
    pipeline_obj = build_pipeline(model_key)
    stages = pipeline_obj.getStages()
    is_lr = model_key.lower() in ('logistic_regression', 'lr', 'logistic regression')

    if is_lr:
        label_counts = train_df.groupBy(LABEL_COL).count().collect()
        total = sum(r['count'] for r in label_counts)
        n_class = len(label_counts)
        weight_dict = {r[LABEL_COL]: total / (n_class * r['count']) for r in label_counts}
        print('Class weights (inverse frequency):')
        for k, v in weight_dict.items(): print(f'  {k} -> {v:.4f}')
        mapping = F.create_map(*[x for kv in weight_dict.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))])
        train_df_w = train_df.withColumn('class_weight', mapping[F.col(LABEL_COL)])
        feat_stages = stages[:-1]
        feat_model = Pipeline(stages=feat_stages).fit(train_df_w)
        train_feat = feat_model.transform(train_df_w).cache()
        test_feat = feat_model.transform(test_df).cache()
        base_lr = stages[-1]; base_lr.setWeightCol('class_weight')
        param_grid = ParamGridBuilder() \
            .addGrid(base_lr.regParam, [0.01, 0.05, 0.1]) \
            .addGrid(base_lr.elasticNetParam, [0.0, 0.15, 0.5]).build()
        evaluator = MulticlassClassificationEvaluator(labelCol='label_index', predictionCol='pred_index', metricName='f1')
        cv = CrossValidator(estimator=base_lr, estimatorParamMaps=param_grid, evaluator=evaluator, numFolds=3, seed=SEED, parallelism=4)
        # parallelism=4 => 4 model fit berjalan paralel (default 1 sangat lambat)
        cv_model = cv.fit(train_feat)
        best_lr = cv_model.bestModel
        print(f'  Best LR params: regParam={best_lr.getRegParam():.4f}, elasticNet={best_lr.getElasticNetParam():.4f}')
        train_pred = best_lr.transform(train_feat)
        test_pred = best_lr.transform(test_feat)
        si_model = feat_model.stages[-1]
        labels = list(si_model.labels)
        when_expr = None
        for i, lbl in enumerate(labels):
            cond = F.col('pred_index').cast('int') == i
            when_expr = F.when(cond, F.lit(lbl)) if when_expr is None else when_expr.when(cond, F.lit(lbl))
        train_pred = train_pred.withColumn('pred_label', when_expr)
        test_pred = test_pred.withColumn('pred_label', when_expr)
        model_obj = cv_model
    else:
        fitted = Pipeline(stages=stages).fit(train_df)
        si_model = fitted.stages[-2]
        labels = list(si_model.labels)
        when_expr = None
        for i, lbl in enumerate(labels):
            cond = F.col('pred_index').cast('int') == i
            when_expr = F.when(cond, F.lit(lbl)) if when_expr is None else when_expr.when(cond, F.lit(lbl))
        train_pred = fitted.transform(train_df).withColumn('pred_label', when_expr)
        test_pred = fitted.transform(test_df).withColumn('pred_label', when_expr)
        model_obj = fitted

    train_metrics = compute_metrics(train_pred)
    test_metrics = compute_metrics(test_pred)
    print(f'  Accuracy : {test_metrics["accuracy"]:.4f}')
    print(f'  F1 Score : {test_metrics["f1_weighted"]:.4f}')
    print(f'  Precision: {test_metrics["precision_weighted"]:.4f}')
    print(f'  Recall   : {test_metrics["recall_weighted"]:.4f}')
    return {'model': model_obj, 'train_prediction': train_pred, 'test_prediction': test_pred,
            'train': train_metrics, 'test': test_metrics}


def save_predictions_to_mongo(predictions_df, model_key: str, collection_name: str) -> None:
    rows = predictions_df\
        .select('comment_id', TEXT_COL, LABEL_COL, 'pred_label')\
        .collect()
    docs = [
        {
            'model': model_key,
            'comment_id': row['comment_id'],
            TEXT_COL: row[TEXT_COL],
            'actual': row[LABEL_COL],
            'predicted': row['pred_label'],
        }
        for row in rows
    ]
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        client[MONGO_DB][collection_name].insert_many(docs)
    print(f'  Prediksi disimpan ke MongoDB: {MONGO_DB}.{collection_name} ({len(docs)} dokumen)')


print('Library dan fungsi berhasil dimuat.')

In [ ]:
spark = create_spark_session('phase2-auto-labeling')
spark.sparkContext.setLogLevel('WARN')
print(f'Spark: {spark.sparkContext.master}')

## 2. Load Info & Retrain Pipeline

In [ ]:
def _normalize_value(val):
    if isinstance(val, (dict, list)):
        return json.dumps(val, ensure_ascii=False, default=str)
    return val

with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    info_doc = client[MONGO_DB]['phase1_best_model'].find_one({'phase': 'phase1'})
if info_doc is None:
    raise RuntimeError('Best model info tidak ditemukan di MongoDB phase1_best_model. Jalankan Phase 1 dulu.')
info = {k: v for k, v in info_doc.items() if not k.startswith('_')}
print(f'Model: {info["best_model_display"]} (F1-macro={info["best_f1_macro"]:.4f})')

docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_LABELED_COLLECTION].find(
        {TEXT_COL: {'$exists': True, '$ne': ''},
         LABEL_COL: {'$exists': True, '$nin': [None, '']}},
        {'_id': 0, 'comment_id': 1, TEXT_COL: 1, LABEL_COL: 1}
    )
    for doc in cursor:
        docs.append({
            'comment_id': doc.get('comment_id', ''),
            TEXT_COL: _normalize_value(doc[TEXT_COL]),
            LABEL_COL: doc.get(LABEL_COL, ''),
        })
train_data = spark.createDataFrame(docs).cache()
print(f'Train data: {train_data.count()} baris')
loaded = build_pipeline(info['best_model_key']).fit(train_data)
print(f'Stages: {[type(s).__name__ for s in loaded.stages]}')

## 3. Load Data Target

In [ ]:
print(f'Membaca dari MongoDB: {MONGO_DB}.{MONGO_PREPROC_COLLECTION}')
docs = []
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    cursor = client[MONGO_DB][MONGO_PREPROC_COLLECTION].find(
        {TEXT_COL: {'$exists': True, '$ne': ''}},
        {'_id': 0, 'comment_id': 1, TEXT_COL: 1}
    )
    for doc in cursor:
        docs.append({
            'comment_id': doc.get('comment_id', ''),
            TEXT_COL: _normalize_value(doc[TEXT_COL]),
        })
udf = spark.createDataFrame(docs).cache()
print(f'Baris: {udf.count()}')

## 4. Prediksi Label

In [ ]:
for c in ('tokens',): 
    if c in udf.columns: udf = udf.drop(c)
udf = udf.withColumn(LABEL_COL, F.lit('dummy'))
pred = loaded.transform(udf).drop(LABEL_COL)
pred = map_prediction_labels(pred, loaded)
print('=== Distribusi Prediksi ===')
pred.groupBy('pred_label').count().orderBy('pred_label').show()
tp = pred.count()
for r in pred.groupBy('pred_label').count().orderBy('pred_label').collect():
    print(f'  {r["pred_label"]:10s}: {r["count"]:5d} ({r["count"]/tp*100:.1f}%)')
pred.select('comment_id', TEXT_COL, 'pred_label').show(5, truncate=60)

## 5. Simpan Hasil Auto-Labeling

In [ ]:
al = pred.select('comment_id', TEXT_COL, F.col('pred_label').alias(LABEL_COL)).withColumn('_source', F.lit('auto_labeled'))

rows = al.collect()
al_docs = [
    {'comment_id': r['comment_id'], TEXT_COL: r[TEXT_COL], LABEL_COL: r[LABEL_COL], '_source': 'auto_labeled'}
    for r in rows
]
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    coll = client[MONGO_DB][MONGO_AUTO_LABELED_COLLECTION]
    coll.drop()
    coll.insert_many(al_docs, ordered=False)
    coll.create_index('comment_id', unique=True)
print(f'Auto-labeled disimpan ke MongoDB: {MONGO_DB}.{MONGO_AUTO_LABELED_COLLECTION} ({len(al_docs)} dokumen)')

stats = {'phase': 'phase2', 'model': info['best_model_display'], 'f1_macro': info['best_f1_macro'],
         'total': tp, 'dist': {str(r['pred_label']): int(r['count']) for r in pred.groupBy('pred_label').count().collect()},
         'timestamp': datetime.now(timezone.utc).isoformat()}
with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
    client[MONGO_DB]['phase2_stats'].replace_one(
        {'phase': 'phase2'}, stats, upsert=True
    )
print(f'Stats saved to MongoDB.')

## 6. Stop Spark

In [ ]:
spark.stop()
print('Phase 2 selesai.')